In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

In [2]:
df = pd.read_csv("../data/processed/operations_cleaned.csv")

In [4]:
df.head(10)

,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,capacity_violation,opening_overflow_tonnes,closing_overflow_tonnes
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,False,0.0,0.0
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,False,0.0,0.0
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,False,0.0,0.0
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,False,0.0,0.0
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,False,0.0,0.0
5,2022-01-06,SITE_001,North,aggressive,CEM_II,30.02,22.19,0.00,22.19,0.00,1.29,14.61,448,False,0.0,0.0
6,2022-01-07,SITE_001,North,aggressive,CEM_I,33.60,33.60,0.00,47.72,14.12,0.90,4.48,448,False,0.0,0.0
7,2022-01-08,SITE_001,North,aggressive,CEM_I,42.12,34.28,14.12,20.16,0.00,1.95,25.92,448,False,0.0,0.0
8,2022-01-09,SITE_001,North,aggressive,CEM_I,0.00,0.00,0.00,34.38,34.38,1.42,16.81,448,False,0.0,0.0
9,2022-01-10,SITE_001,North,aggressive,CEM_II,30.99,30.99,34.38,35.37,38.76,3.49,16.57,448,False,0.0,0.0


In [3]:
df_copy = df.copy()
df_copy['date'] = pd.to_datetime(df_copy['date'])

df_copy['week'] = df_copy['date'].dt.isocalendar().week

df_copy


,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,capacity_violation,opening_overflow_tonnes,closing_overflow_tonnes,week
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,False,0.0,0.0,52
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,False,0.0,0.0,52
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,False,0.0,0.0,1
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,False,0.0,0.0,1
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,False,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32875,2024-12-27,SITE_030,South,aggressive,CEM_III,58.48,29.63,0.00,29.63,0.00,7.17,12.00,316,False,0.0,0.0,52
32876,2024-12-28,SITE_030,South,aggressive,CEM_I,45.39,22.35,0.00,22.35,0.00,2.13,8.04,316,False,0.0,0.0,52
32877,2024-12-29,SITE_030,South,aggressive,CEM_III,58.47,14.21,0.00,14.21,0.00,0.28,2.26,316,False,0.0,0.0,52
32878,2024-12-30,SITE_030,South,aggressive,CEM_II,59.58,10.65,0.00,10.65,0.00,4.58,9.55,316,False,0.0,0.0,1


In [4]:
import pandas as pd

# Load data

df_copy['date'] = pd.to_datetime(df['date'])
df_copy.set_index('date', inplace=True)

# Define aggregation rules for each column
agg_rules = {
    'planned_pour_tonnes': 'sum',
    'consumed_tonnes': 'sum',
    'opening_inventory_tonnes': 'first',  # or 'mean'
    'deliveries_tonnes': 'sum',
    'closing_inventory_tonnes': 'last',   # or 'mean'
    'rain_mm': 'sum',
    'avg_temp_c': 'mean',
    'silo_capacity': 'first',  # constant per site
    'capacity_violation': 'any',
    'opening_overflow_tonnes': 'sum',
    'closing_overflow_tonnes': 'sum',
    # For categorical columns, take the first/mode
    'site_id': 'first',
    'region': 'first',
    'behavior': 'first',
    'cement_type': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]
}

# Resample to weekly (Monday-Sunday) and aggregate
weekly_df = df_copy.groupby('site_id').resample('W-MON').agg(agg_rules).reset_index()


AttributeError: 'any' is not a valid function for 'DatetimeIndexResamplerGroupby' object